# Importing Libraries

In [ ]:
import joblib
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import contractions
from datasets import load_dataset
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, LSTM, Bidirectional, Concatenate, Dense, Attention, AdditiveAttention
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import load_model

# Importing Dataset

In [ ]:
dataset = load_dataset("cnn_dailymail", "3.0.0")
train = pd.DataFrame(dataset['train']).sample(50000, random_state=42)
val = pd.DataFrame(dataset['validation']).sample(5000, random_state=42)
test = pd.DataFrame(dataset['test']).sample(5000, random_state=42)

# Data Preprocessing

In [ ]:
for df in [train, val]:
    df['article'] = df['article'].apply(lambda x: x.lower())
    df['article'] = df['article'].apply(lambda x: contractions.fix(x))
    df['article'] = df['article'].apply(lambda x: re.sub(r"([?.!,])", r" \1 ", x))
    df['article'] = df['article'].apply(lambda x: re.sub(r'[^a-zA-Z0-9?.!, ]', '', x))
    df['article'] = df['article'].apply(lambda x: " ".join(x.split()))

    df['highlights'] = df['highlights'].apply(lambda x: x.lower())
    df['highlights'] = df['highlights'].apply(lambda x: contractions.fix(x))
    df['highlights'] = df['highlights'].apply(lambda x: re.sub(r"([?.!,])", r" \1 ", x))
    df['highlights'] = df['highlights'].apply(lambda x: 'sostoken ' + x + ' eotoken')
    df['highlights'] = df['highlights'].apply(lambda x: re.sub(r'[^a-zA-Z0-9?.!, ]', '', x))
    df['highlights'] = df['highlights'].apply(lambda x: " ".join(x.split()))

In [ ]:
max_art_len = 500
max_sum_len = 70

train = train[train['article'].apply(lambda x: len(x.split()) <= max_art_len)]
train = train[train['highlights'].apply(lambda x: len(x.split()) <= max_sum_len)]
val = val[val['article'].apply(lambda x: len(x.split()) <= max_art_len)]
val = val[val['highlights'].apply(lambda x: len(x.split()) <= max_sum_len)]

In [ ]:
x_voc_limit = 30000
y_voc_limit = 15000

x_tokenizer = Tokenizer(num_words=x_voc_limit)
x_tokenizer.fit_on_texts(list(train['article']))

y_tokenizer = Tokenizer(num_words=y_voc_limit)
y_tokenizer.fit_on_texts(list(train['highlights']))

x_tr_seq = x_tokenizer.texts_to_sequences(train['article'])
x_val_seq = x_tokenizer.texts_to_sequences(val['article'])
y_tr_seq = y_tokenizer.texts_to_sequences(train['highlights'])
y_val_seq = y_tokenizer.texts_to_sequences(val['highlights'])

x_train = pad_sequences(x_tr_seq, maxlen=max_art_len, padding='post')
x_val = pad_sequences(x_val_seq, maxlen=max_art_len, padding='post')
y_train = pad_sequences(y_tr_seq, maxlen=max_sum_len, padding='post')
y_val = pad_sequences(y_val_seq, maxlen=max_sum_len, padding='post')

x_voc = x_tokenizer.num_words + 1
y_voc = y_tokenizer.num_words + 1

In [ ]:
latent_dim = 256
embedding_dim = 128

decoder_input_data = y_train[:, :-1]
decoder_target_data = y_train.reshape(y_train.shape[0], y_train.shape[1], 1)[:, 1:]
decoder_input_val = y_val[:, :-1]
decoder_target_val = y_val.reshape(y_val.shape[0], y_val.shape[1], 1)[:, 1:]

# Encoder Decoder

In [ ]:
encoder_inputs = Input(shape=(max_art_len,))
encoder_embedding = Embedding(x_voc, embedding_dim, trainable=True)(encoder_inputs)
encoder_lstm = Bidirectional(LSTM(latent_dim, return_sequences=True, return_state=True))
encoder_outputs, forward_h, forward_c, backward_h, backward_c = encoder_lstm(encoder_embedding)

state_h = Concatenate()([forward_h, backward_h])
state_c = Concatenate()([forward_c, backward_c])
encoder_states = [state_h, state_c]

decoder_inputs = Input(shape=(None,))
decoder_embedding = Embedding(y_voc, embedding_dim, trainable=True)(decoder_inputs)
decoder_lstm = LSTM(latent_dim * 2, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)

decoder_dense = Dense(y_voc, activation='softmax')
decoder_outputs_normal = decoder_dense(decoder_outputs)

model_normal = Model([encoder_inputs, decoder_inputs], decoder_outputs_normal)
model_normal.compile(optimizer='rmsprop', loss='sparse_categorical_crossentropy')

In [ ]:
model_normal.summary()

In [ ]:
history_normal = model_normal.fit(
    [x_train, decoder_input_data], decoder_target_data,
    epochs=10, batch_size=128,
    validation_data=([x_val, decoder_input_val], decoder_target_val)
)

In [ ]:
model_normal.save('model_normal.keras')

joblib.dump(x_tokenizer, 'x_tokenizer.joblib')
joblib.dump(y_tokenizer, 'y_tokenizer.joblib')

del model_normal
gc.collect()
tf.keras.backend.clear_session()

# Encoder Decoder with Badhanau Attention

In [ ]:
encoder_inputs = Input(shape=(max_art_len,))
encoder_embedding = Embedding(x_voc, embedding_dim, trainable=True)(encoder_inputs)
encoder_lstm = Bidirectional(LSTM(latent_dim, return_sequences=True, return_state=True))
encoder_outputs, forward_h, forward_c, backward_h, backward_c = encoder_lstm(encoder_embedding)

state_h = Concatenate()([forward_h, backward_h])
state_c = Concatenate()([forward_c, backward_c])
encoder_states = [state_h, state_c]

decoder_inputs = Input(shape=(None,))
decoder_embedding = Embedding(y_voc, embedding_dim, trainable=True)(decoder_inputs)
decoder_lstm = LSTM(latent_dim * 2, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)

attn_layer_bahdanau = AdditiveAttention(name='bahdanau_attention')
query_value_attention_seq = attn_layer_bahdanau([decoder_outputs, encoder_outputs])
decoder_concat_bahdanau = Concatenate(axis=-1)([decoder_outputs, query_value_attention_seq])

decoder_dense = Dense(y_voc, activation='softmax')
decoder_outputs_bahdanau = decoder_dense(decoder_concat_bahdanau)

model_bahdanau = Model([encoder_inputs, decoder_inputs], decoder_outputs_bahdanau)
model_bahdanau.compile(optimizer='rmsprop', loss='sparse_categorical_crossentropy')

In [ ]:
model_bahdanau.summary()

In [ ]:
history_bahdanau = model_bahdanau.fit(
    [x_train, decoder_input_data], decoder_target_data,
    epochs=10, batch_size=128,
    validation_data=([x_val, decoder_input_val], decoder_target_val)
)

In [ ]:
model_bahdanau.save('model_bahdanau.keras')

del model_bahdanau
gc.collect()
tf.keras.backend.clear_session()

# Encoder Decoder with Luong Attention


In [ ]:
encoder_inputs = Input(shape=(max_art_len,))
encoder_embedding = Embedding(x_voc, embedding_dim, trainable=True)(encoder_inputs)
encoder_lstm = Bidirectional(LSTM(latent_dim, return_sequences=True, return_state=True))
encoder_outputs, forward_h, forward_c, backward_h, backward_c = encoder_lstm(encoder_embedding)

state_h = Concatenate()([forward_h, backward_h])
state_c = Concatenate()([forward_c, backward_c])
encoder_states = [state_h, state_c]

decoder_inputs = Input(shape=(None,))
decoder_embedding = Embedding(y_voc, embedding_dim, trainable=True)(decoder_inputs)
decoder_lstm = LSTM(latent_dim * 2, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)

attention_layer = Attention(name='luong_attention')
attention_outputs = attention_layer([decoder_outputs, encoder_outputs])
decoder_luong_concat = Concatenate(axis=-1, name='luong_concat')([decoder_outputs, attention_outputs])

decoder_dense = Dense(y_voc, activation='softmax')
decoder_outputs_luong = decoder_dense(decoder_luong_concat)

model_luong = Model([encoder_inputs, decoder_inputs], decoder_outputs_luong)
model_luong.compile(optimizer='rmsprop', loss='sparse_categorical_crossentropy')

In [ ]:
model_luong.summary()

In [ ]:
history_luong = model_luong.fit(
    [x_train, decoder_input_data], decoder_target_data,
    epochs=10, batch_size=128,
    validation_data=([x_val, decoder_input_val], decoder_target_val)
)

In [ ]:
model_luong.save('model_luong.keras')

del model_luong
gc.collect()
tf.keras.backend.clear_session()

# Summarizations

In [ ]:
reverse_target_word_index = y_tokenizer.index_word
target_word_index = y_tokenizer.word_index

def generate_summary(model_filename, input_seq):
    tf.keras.backend.clear_session()
    model = load_model(model_filename, custom_objects={'Attention': Attention, 'AdditiveAttention': AdditiveAttention})

    bidi_lstm = [layer for layer in model.layers if isinstance(layer, Bidirectional)][0]
    dec_emb_layer = [layer for layer in model.layers if isinstance(layer, Embedding)][1]
    dec_lstm_layer = [layer for layer in model.layers if isinstance(layer, LSTM) and not isinstance(layer, Bidirectional)][0]
    dense_layer = [layer for layer in model.layers if isinstance(layer, Dense)][0]

    attn_layer = None
    for layer in model.layers:
        if isinstance(layer, (Attention, AdditiveAttention)):
            attn_layer = layer
            break

    enc_inputs = model.input[0]
    enc_out, f_h, f_c, b_h, b_c = bidi_lstm.output
    state_h = Concatenate()([f_h, b_h])
    state_c = Concatenate()([f_c, b_c])
    encoder_model = Model(inputs=enc_inputs, outputs=[enc_out, state_h, state_c])

    dec_inputs = model.input[1]
    dec_state_input_h = Input(shape=(latent_dim * 2,))
    dec_state_input_c = Input(shape=(latent_dim * 2,))
    dec_hidden_state_input = Input(shape=(max_art_len, latent_dim * 2))

    dec_emb = dec_emb_layer(dec_inputs)
    dec_out, state_h2, state_c2 = dec_lstm_layer(dec_emb, initial_state=[dec_state_input_h, dec_state_input_c])

    if attn_layer:
        concat_layer = [layer for layer in model.layers if isinstance(layer, Concatenate)][-1]
        attn_out = attn_layer([dec_out, dec_hidden_state_input])
        dec_concat = concat_layer([dec_out, attn_out])
        final_out = dense_layer(dec_concat)

        decoder_model = Model(
            [dec_inputs] + [dec_hidden_state_input, dec_state_input_h, dec_state_input_c],
            [final_out] + [state_h2, state_c2]
        )
    else:
        final_out = dense_layer(dec_out)
        decoder_model = Model(
            [dec_inputs] + [dec_state_input_h, dec_state_input_c],
            [final_out] + [state_h2, state_c2]
        )

    e_out, e_h, e_c = encoder_model.predict(input_seq, verbose=0)
    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = target_word_index['sostoken']

    stop_condition = False
    decoded_sentence = ''

    while not stop_condition:
        if attn_layer:
            output_tokens, h, c = decoder_model.predict([target_seq] + [e_out, e_h, e_c], verbose=0)
        else:
            output_tokens, h, c = decoder_model.predict([target_seq] + [e_h, e_c], verbose=0)

        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_token = reverse_target_word_index.get(sampled_token_index, '')

        if sampled_token != 'eotoken':
            decoded_sentence += ' ' + sampled_token

        if sampled_token == 'eotoken' or len(decoded_sentence.split()) >= (max_sum_len - 1):
            stop_condition = True

        target_seq = np.zeros((1, 1))
        target_seq[0, 0] = sampled_token_index
        e_h, e_c = h, c

    del model, encoder_model, decoder_model
    gc.collect()

    return decoded_sentence.strip()

# Camparisons

In [ ]:
sample_idx = 42
test_article = x_val[sample_idx].reshape(1, max_art_len)

print("--- ORIGINAL ARTICLE ---")
print(val['article'].values[sample_idx][:800] + "...\n")

print("--- ACTUAL SUMMARY ---")
print(val['highlights'].values[sample_idx] + "\n")

saved_models = ['model_normal.keras', 'model_bahdanau.keras', 'model_luong.keras']

for model_file in saved_models:
    print(f"Generating summary using {model_file}...")
    try:
        summary = generate_summary(model_file, test_article)
        print(f"PREDICTED: {summary}\n")
    except Exception as e:
        print(f"Could not generate summary for {model_file}. Error: {e}\n")

plt.figure(figsize=(10, 5))
plt.plot(history_normal.history['val_loss'], label='Normal Val Loss')
plt.plot(history_bahdanau.history['val_loss'], label='Bahdanau Val Loss')
plt.plot(history_luong.history['val_loss'], label='Luong Val Loss')
plt.title('Validation Loss Comparison')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()